In [2]:
import os
import re
from typing import cast

os.environ["TRANSFORMERS_VERBOSITY"] = "info"
# NOTE: expandable_segments is incompatible with vLLM's CuMemAllocator (sleep mode)
# See: https://github.com/pytorch/pytorch/issues/147851

from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer
from peft import LoraConfig
from datasets import DatasetDict, load_dataset

model_name = "ibm-granite/granite-4.0-350m"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


INFO 03-10 21:55:36 [__init__.py:216] Automatically detected platform cuda.


In [3]:
SYSTEM_PROMPT = (
    "You are a helpful AI Assistant. Answer the user's question step by step, showing your work. "
    "Be concise but complete.\n"
    "Always include the final answer at the end, after '####'. The final answer should be a single number.\n"
)


def make_messages(question: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

## Train

In [4]:
def extract_gold(example: dict) -> str:
    answer = example["answer"]
    match = re.search(r"####\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Could not extract gold answer from: {answer!r}")
    return match.group(1).replace(",", "")


ds = cast(DatasetDict, load_dataset("openai/gsm8k", "main"))

ds["train"] = ds["train"].map(
    lambda ex: {"gold": extract_gold(ex), "prompt": make_messages(ex["question"])}
)
ds["test"] = ds["test"].map(
    lambda ex: {"gold": extract_gold(ex), "prompt": make_messages(ex["question"])}
)

In [5]:
def reward_correctness(prompts, completions, gold, **kwargs) -> list[float]:
    return [5.0 if g in c else 0.0 for g, c in zip(gold, completions)]


def reward_brevity(prompts, completions, **kwargs) -> list[float]:
    scores = []
    for c in completions:
        length = len(c)
        if length < 100:
            scores.append(-0.01 * (100 - length) + 1)
        elif length > 300:
            scores.append(-0.012 * (length - 300) + 1)
        else:
            scores.append(1.0)
    return scores

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

peft_config = LoraConfig()

config = GRPOConfig(
    output_dir="./grpo-output",
    num_generations=2,
    beta=0.01,
    max_prompt_length=256,
    max_completion_length=128,
    learning_rate=1e-4,
    temperature=0.7,
    use_vllm=True,
    vllm_mode="colocate",
    use_liger_kernel=True,
    vllm_enable_sleep_mode=True,
    vllm_gpu_memory_utilization=0.3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,  # keeps effective batch size at 4
    num_train_epochs=1,
    logging_steps=1,
    gradient_checkpointing=True,
)

In [ ]:
trainer = GRPOTrainer(
    model=model_name,
    args=config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    reward_funcs=[reward_correctness, reward_brevity],
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
import torch

model = trainer.model
model.eval()

with torch.inference_mode():
    messages = make_messages(ds["test"][0]["question"])

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    print(tokenizer.decode(out[0], skip_special_tokens=False))

In [ ]:
tokenizer.decode(out[0], skip_special_tokens=False).split("<|end_of_role|>")[-1].split(
    "<|end_of_text|>"
)[0].strip()